# ScholarGuard — Stage 2 exploration

Visual debugging playground for the copy-move detector. Run cells top to bottom.

Pipeline: SIFT self-matching → offset-space DBSCAN → RANSAC affine verification → ZNCC region growing → confidence score.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # make `src` importable from notebooks/

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.detectors.copy_move_detector import CopyMoveDetector, DetectorConfig
from src.utils.image_io import load_image, load_mask, list_images, find_ground_truth_mask
from src.utils.visualization import side_by_side, overlay_mask
from src.utils.synth import make_base_figure, apply_copy_move
from src.evaluation.metrics import compute_pixel_metrics

def show(img, title="", figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap="gray")
    plt.title(title); plt.axis("off"); plt.show()

## 1. Make (or load) a forged sample and run the detector

In [ ]:
# Either generate a fresh forgery on the fly...
rng = np.random.default_rng(7)
forged, gt_mask = apply_copy_move(make_base_figure(rng), rng, patch_size=(80, 100))

# ...or load one from the dataset:
# forged = load_image("../data/synthetic/forged_000.png")
# gt_mask = load_mask("../data/synthetic/forged_000_mask.png")

detector = CopyMoveDetector()
result = detector.detect(forged)
print(f"forged={result['forged']}  confidence={result['confidence']}  regions={len(result['regions'])}")
print(compute_pixel_metrics(result["mask"], gt_mask))
show(result["visualization"], "detection (green=src, red=dup)")

In [ ]:
show(side_by_side(forged, result["mask"], gt_mask), "original | predicted | ground truth", figsize=(16, 5))

## 2. Inspect intermediate pipeline stages

In [ ]:
gray = cv2.cvtColor(forged, cv2.COLOR_BGR2GRAY)
keypoints, descriptors = detector._extract_features(gray)
matches = detector._self_match(keypoints, descriptors, gray.shape)
clusters = detector._cluster_and_verify(matches, keypoints)
print(f"keypoints={len(keypoints)}  ratio-test matches={len(matches)}  verified clusters={len(clusters)}")

# Draw all candidate matches (yellow) and verified inliers (green)
canvas = cv2.drawKeypoints(forged, keypoints, None, (180, 180, 180))
pts = np.array([kp.pt for kp in keypoints])
for i, j in matches:
    p1, p2 = tuple(pts[i].astype(int)), tuple(pts[j].astype(int))
    cv2.line(canvas, p1, p2, (0, 220, 220), 1, cv2.LINE_AA)
for c in clusters:
    for s, d in zip(c.src_pts, c.dst_pts):
        cv2.line(canvas, tuple(s.astype(int)), tuple(d.astype(int)), (0, 200, 0), 2, cv2.LINE_AA)
show(canvas, "self-matches: yellow=candidates, green=RANSAC inliers")

## 3. Sanity check on a clean image

In [ ]:
clean = make_base_figure(np.random.default_rng(11))
clean_result = detector.detect(clean)
print(f"forged={clean_result['forged']}  confidence={clean_result['confidence']}")
show(clean_result["visualization"], "clean image — expect no detections")

## 4. Batch check over the dataset

Full benchmarking lives in `src/evaluation/metrics.py`; this cell is for quick eyeballing of a few images.

In [ ]:
for path in list_images("../data/synthetic")[:4]:
    image = load_image(path)
    gt_path = find_ground_truth_mask(path)
    gt = load_mask(gt_path) if gt_path else None
    r = detector.detect(image)
    title = f"{os.path.basename(path)}  forged={r['forged']}  conf={r['confidence']}"
    if gt is not None:
        title += f"  IoU={compute_pixel_metrics(r['mask'], gt)['iou']:.2f}"
    show(side_by_side(image, r["mask"], gt), title, figsize=(15, 4))

## 5. Experiment: tweak the config

All detector knobs live in `DetectorConfig` — try changing thresholds and re-running section 1.

In [ ]:
custom = CopyMoveDetector(DetectorConfig(zncc_threshold=0.8, cluster_eps=16))
r = custom.detect(forged)
print(f"forged={r['forged']}  confidence={r['confidence']}")